# ***Analisis de robos generales en Republica Dominicana 2024-2025*** 📈📉📊

In [313]:
# Librerias que seran utilizadas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Para el analisis predictivo
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Informacion de los datos:

Este conjunto de datos contiene las estadísticas de Robos de Automotores, Armas de fuego y Denuncias de Robos Generales en el Ministerio de Interior y Policía (MIP), generadas durante el año 2018 y 2026, donde se puede encontrar el registro de robos.

In [314]:
# exportamos los datos
df = pd.DataFrame(pd.read_excel(r'/content/robos-automotor-armas-de-fuego-y-denuncias-de-robo-2018-2025.xlsx', sheet_name= 'ROBOS_GENERALES'))

In [315]:
df

,AÑOS,MESES,PROVINCIAS,TIPO DE ROBO,CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES
0,2024,ene,Azua,Arrebato,5
1,2024,ene,Azua,Asalto (Atraco),6
2,2024,ene,Azua,Robo Simple,30
3,2024,ene,Azua,Rotura (Escalamiento),19
4,2024,ene,Baoruco,Arrebato,2
...,...,...,...,...,...
2959,2025,sep,Santo Domingo,Rotura A Vehículo,37
2960,2025,sep,Valverde,Asalto (Atraco),6
2961,2025,sep,Valverde,Robo Simple,5
2962,2025,sep,Valverde,Rotura A Negocio,4


## Conocer nuestros dataset

In [316]:
print(df.columns)

Index(['AÑOS', 'MESES', 'PROVINCIAS', 'TIPO DE ROBO',
       'CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES'],
      dtype='object')


In [317]:
print(df.shape)

(2964, 5)


In [318]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964 entries, 0 to 2963
Data columns (total 5 columns):
 #   Column                                       Non-Null Count  Dtype 
---  ------                                       --------------  ----- 
 0   AÑOS                                         2964 non-null   int64 
 1   MESES                                        2964 non-null   object
 2   PROVINCIAS                                   2964 non-null   object
 3   TIPO DE ROBO                                 2964 non-null   object
 4   CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES  2964 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 115.9+ KB
None


In [319]:
print(df.describe())

              AÑOS  CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES
count  2964.000000                                  2964.000000
mean   2024.473347                                    47.849190
std       0.499373                                   142.046552
min    2024.000000                                     1.000000
25%    2024.000000                                     2.000000
50%    2024.000000                                     8.000000
75%    2025.000000                                    28.000000
max    2025.000000                                  1429.000000


In [320]:
# Encontra valores unicos
for column in df.columns:
    unique_values = df[column].unique()
    print(f"Valores únicos en la columna '{column}':")
    print(unique_values)
    print()

Valores únicos en la columna 'AÑOS':
[2024 2025]

Valores únicos en la columna 'MESES':
['ene' 'feb' 'mar' 'abr' 'may' 'jun' 'jul' 'ago' 'sep' 'oct' 'nov' 'dic']

Valores únicos en la columna 'PROVINCIAS':
['Azua' 'Baoruco' 'Barahona' 'Dajabón' 'Distrito Nacional' 'Duarte'
 'El Seibo' 'Elías Piña' 'Espaillat' 'Hato Mayor' 'Hermanas Mirabal'
 'Independencia' 'La Altagracia' 'La Romana' 'La Vega'
 'María Trinidad Sánchez' 'Monseñor Nouel' 'Monte Cristi' 'Monte Plata'
 'Pedernales' 'Peravia' 'Puerto Plata' 'Samaná' 'San Cristóbal'
 'San José de Ocoa' 'San Juan' 'San Pedro de Macorís' 'Sanchez Ramírez'
 'Santiago' 'Santiago Rodríguez' 'Santo Domingo' 'Valverde' '#N/D']

Valores únicos en la columna 'TIPO DE ROBO':
['Arrebato' 'Asalto (Atraco)' 'Robo Simple' 'Rotura (Escalamiento)'
 'Robo Agravado' 'Rotura A Residencia' 'Rotura A Negocio'
 'Rotura A Vehículo']

Valores únicos en la columna 'CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES':
[   5    6   30   19    2    1   29    3   10  118   11

# 1) Limpieza de los datos para que no tenga errores en nuestros analisis.

In [321]:
# Trabajando con la copia del dataset
df_copy = df.copy()
# Trabajando con la variable
df_copy['MESES'] = df_copy['MESES'].str.upper()
# Creacion de variable numerica del mes
df_copy['MESES_NUM'] = df_copy['MESES'].map({
    'ENE': 1,
    'FEB': 2,
    'MAR': 3,
    'ABR': 4,
    'MAY': 5,
    'JUN': 6,
    'JUL': 7,
    'AGO': 8,
    'SEP': 9,
    'OCT': 10,
    'NOV': 11,
    'DIC': 12
})

In [322]:
# Encontrar valores vacio o duplicados en nuestros dataset
for column in df_copy.columns:
    missing_values = df_copy[column].isnull().sum()
    duplicate_values = df_copy[column].duplicated().sum()
    print(f"Valores faltantes en la columna '{column}': {missing_values}")
    print(f"Valores duplicados en la columna '{column}': {duplicate_values}")
    print()

Valores faltantes en la columna 'AÑOS': 0
Valores duplicados en la columna 'AÑOS': 2962

Valores faltantes en la columna 'MESES': 0
Valores duplicados en la columna 'MESES': 2952

Valores faltantes en la columna 'PROVINCIAS': 0
Valores duplicados en la columna 'PROVINCIAS': 2931

Valores faltantes en la columna 'TIPO DE ROBO': 0
Valores duplicados en la columna 'TIPO DE ROBO': 2956

Valores faltantes en la columna 'CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES': 0
Valores duplicados en la columna 'CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES': 2660

Valores faltantes en la columna 'MESES_NUM': 0
Valores duplicados en la columna 'MESES_NUM': 2952



In [323]:
# Valores de variables tipo de robo
df_copy['TIPO DE ROBO'].unique()

array(['Arrebato', 'Asalto (Atraco)', 'Robo Simple',
       'Rotura (Escalamiento)', 'Robo Agravado', 'Rotura A Residencia',
       'Rotura A Negocio', 'Rotura A Vehículo'], dtype=object)

In [324]:
# Normalizacion de la variable tipo robo

normalizacion = {
 'Arrebato': 'Robo directo',
 'Asalto (Atraco)': 'Robo directo',
 'Robo Simple': 'Robo general',
 'Robo Agravado': 'Robo general',
 'Rotura (Escalamiento)': 'Robo por rotura',
 'Rotura A Residencia': 'Robo por rotura',
 'Rotura A Negocio': 'Robo por rotura',
 'Rotura A Vehículo': 'Robo por rotura'
}

df_copy['TIPO DE ROBO'] = df_copy['TIPO DE ROBO'].map(normalizacion) # De esta forma tenemos 3 categorias de los tipos de robos donde las demas permanecen la misma

In [325]:
# Variable provincia con datos '#N/D'
df_copy[df_copy['PROVINCIAS'] == '#N/D']

,AÑOS,MESES,PROVINCIAS,TIPO DE ROBO,CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES,MESES_NUM
1718,2025,ENE,#N/D,Robo directo,12,1
1719,2025,ENE,#N/D,Robo general,391,1
1874,2025,FEB,#N/D,Robo directo,13,2
1875,2025,FEB,#N/D,Robo general,286,2
2025,2025,MAR,#N/D,Robo directo,14,3
2026,2025,MAR,#N/D,Robo general,379,3
2027,2025,MAR,#N/D,Robo por rotura,1,3
2178,2025,ABR,#N/D,Robo directo,6,4
2179,2025,ABR,#N/D,Robo general,100,4
2337,2025,MAY,#N/D,Robo directo,5,5


In [326]:
# Pasar nuestra variable año a int
df_copy['AÑOS'] = df_copy['AÑOS'].astype(int)

In [327]:
# Remplazamos los datos '#N/D' por desconocido
for column in df_copy.columns:
    df_copy[column] = df_copy[column].replace('#N/D', 'Desconocido')

# 2) *`Analisis descriptivo`* 📊📉

In [328]:
# Provincias top por año
df_copy.groupby(['AÑOS', 'PROVINCIAS'])['CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES'].sum().sort_values(ascending=False).head(100)

AÑOS  PROVINCIAS        
2024  Santo Domingo         31553
2025  Santo Domingo         20618
2024  Distrito Nacional     14286
2025  Distrito Nacional      8356
2024  Santiago               7136
                            ...  
      Independencia           142
2025  Elías Piña              128
2024  Elías Piña               90
      Santiago Rodríguez       57
2025  Independencia            48
Name: CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES, Length: 65, dtype: int64

In [329]:
# Ahora graficamos nuestras provincias con mas cantidad de robos
conteo = (
    df_copy.groupby(['AÑOS', 'PROVINCIAS'])['CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES']
    .sum()
    .reset_index(name='conteo')
)

# Utilizamos la libreria plotly
import plotly.express as px

fig = px.bar(conteo, x='PROVINCIAS', y='conteo', color='AÑOS', barmode='group',
             title= 'Robos de 2024 - 2025 por Provincia')
fig.show()

In [330]:
# Grafico de valores atipicos por tipos de robos
px.box(df_copy, x='TIPO DE ROBO', y='CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES', title='Grafico de valores atipicos por tipos de robos')

## `Explicacion de la grafica`

Se observa que las provincias con mayor cantidad de robos en la República Dominicana son Santo Domingo, Santiago y el Distrito Nacional, destacándose especialmente Santo Domingo con valores considerablemente superiores al resto.

Esta diferencia evidencia una alta concentración de robos en un número reducido de provincias, mientras que las demás presentan niveles significativamente más bajos.

Es importante señalar que estos resultados corresponden a valores absolutos, por lo que podrían estar influenciados por factores como la densidad poblacional y la actividad económica de cada provincia.

## `Medidas a tomar:`

Se recomienda implementar una distribución estratégica de recursos de seguridad, enfocada en:

Provincias con mayor incidencia delictiva
Zonas urbanas de alta densidad
Áreas comerciales

Además, sería conveniente realizar análisis más detallados que permitan identificar:

Sectores específicos dentro de cada provincia
Tipos de robo predominantes
Patrones temporales

In [331]:
# Grafico de linea de cantidad de robos por años poniendolo en comparacion
orden_meses = ['ENE', 'FEB', 'MAR', 'ABR', 'MAY', 'JUN',
               'JUL', 'AGO', 'SEP', 'OCT', 'NOV', 'DIC']

linea = (
    df_copy.groupby(['AÑOS', 'MESES'])['CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES']
    .sum()
    .reset_index()
)

linea['MESES'] = pd.Categorical(
    linea['MESES'],
    categories=orden_meses,
    ordered=True
)

linea = linea.sort_values('MESES')



figura_2 = px.line(
    linea,
    x='MESES',
    y='CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES',
    color='AÑOS',
    markers=True,
    title='Comparación mensual de robos (2024 vs 2025)'
)

figura_2.update_layout(
    xaxis_title='Mes',
    yaxis_title='Cantidad de robos',
    hovermode='x unified'
)

figura_2.show()

In [332]:
# Promedio de robos por mes
df_copy.groupby(['MESES','TIPO DE ROBO'])['CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES'].mean().sort_values(ascending=False)

,,CANTIDAD DE DENUNCIAS POR ROBO DE GENERALES
MESES,TIPO DE ROBO,
ENE,Robo general,110.253165
MAR,Robo general,109.455696
ABR,Robo general,108.226667
JUN,Robo general,106.373333
MAY,Robo general,105.384615
AGO,Robo general,104.430556
FEB,Robo general,103.315789
JUL,Robo general,101.987013
SEP,Robo general,96.042254


## `Explicacion de la grafica`
Se observa que durante el año 2024 se registró una mayor cantidad de robos en comparación con 2025 en la mayoría de los meses analizados. No obstante, es importante considerar que los datos de 2025 no están completos, lo que limita una comparación definitiva.

A nivel mensual, se identifica que meses como enero y marzo presentan los valores más altos de robos, manteniéndose como picos en los distintos tipos de delitos analizados.

Este comportamiento podría estar asociado a períodos de mayor actividad social y económica, aunque esta relación no se evalúa directamente en los datos.

## `Medidas a tomar:`

Se recomienda reforzar la seguridad en los meses donde se observa mayor incidencia de robos, especialmente en:

Enero
Marzo

Esto puede incluir:

Mayor presencia policial
Operativos preventivos
Vigilancia en zonas de alta concurrencia

Asimismo, sería recomendable profundizar el análisis incorporando variables adicionales que permitan explicar con mayor precisión las causas de estos incrementos.
